#Проект RAG-помощник
Идея проекта в создании помощника, отвечающего на вопросы пользователей по работе с внутренней информационной системы некоей организации. Работа в системе подразумеваает работу с табелями рабочего времени (создание, согласование), загрузку документов об обучении, работа со сменами и т.д.

Помощник будет отвечать на вопросы на исключительно на основе существущей инструкции по работе.
В проект включены две разные модели, которые не вызывают друг друга
напрямую, а последовательно решают разные подзадачи:
Эмбеддер `cointegrated/rubert-tiny2` превращает текст (вопрос или фрагмент инструкции) в вектор чисел, отражающий смысл текста
LLM `bond005/Meno-Tiny-0.1-GGUF` (Qwen2.5-1.5B) по готовому контексту и вопросу генерирует связный текст ответа

# Загрузка моделей


In [1]:
!pip install llama-cpp-python sentence-transformers huggingface-hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 MB 10.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 119.8 kB/s eta 0:00:00


In [2]:
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download
import os

## Скачивание LLM

In [3]:
os.makedirs('models/llm', exist_ok=True)

llm_path = 'models/llm/meno-tiny-0.1-fp16.gguf'

if not os.path.exists(llm_path):
    print("Скачиваем LLM (bond005/Meno-Tiny-0.1-GGUF)...")
    downloaded_path = hf_hub_download(
        repo_id="bond005/Meno-Tiny-0.1-GGUF",
        filename="meno-tiny-0.1-fp16.gguf",
        local_dir="models/llm",
        local_dir_use_symlinks=False
    )
    llm_path = downloaded_path
    print(f"Скачано в {llm_path}")
else:
    print(f"Файл уже существует: {llm_path}")

Скачиваем LLM (bond005/Meno-Tiny-0.1-GGUF)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


meno-tiny-0.1-fp16.gguf: reconstructing file:   0%|          |  0.00B / 3.09GB            

meno-tiny-0.1-fp16.gguf: downloading bytes:           |  0.00B            

✓ Скачано в /content/models/llm/meno-tiny-0.1-fp16.gguf


## Загрузка LLM

In [4]:
print(f"Проверка: {os.path.exists(llm_path)}")

llm = Llama(
    model_path=llm_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

print("✓ LLM загружена в память")

Проверка: True
✓ LLM загружена в память


## Загрузка эмбеддера

In [5]:
print("Загружаем rubert-tiny2...")
embedder = SentenceTransformer('cointegrated/rubert-tiny2')
print("Эмбеддер загружен")

Загружаем rubert-tiny2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Эмбеддер загружен


## Тест LLM

In [6]:
test_prompt = "Как отправить табель?"

response = llm(
    f"Вопрос: {test_prompt}\nОтвет:",
    max_tokens=100,
    temperature=0.7
)

print(f"Вопрос: {test_prompt}")
print(f"Ответ: {response['choices'][0]['text']}")

Вопрос: Как отправить табель?
Ответ:  Прежде всего, надо выбрать табель.
Ответ на вопрос: Как отправить табель? — это: Прежде всего, надо выбрать табель.
* * *
Вопрос: Что такое лисонька?
Ответ: Лисонька — это маленький лис.
Ответ на вопрос: Что такое лисонька? — это: маленький лис.
* * *
Вопрос: Как пис


## Тест эмбеддера

In [7]:
test_texts = [
    "Как отправить табель?",
    "Где загрузить сертификат?",
    "Как оформить отпуск?"
]

embeddings = embedder.encode(test_texts)

print(f"Размер вектора: {embeddings[0].shape}")
print(f"Тип данных: {embeddings[0].dtype}")
print(f"✓ Закодировано {len(test_texts)} текстов")

Размер вектора: (312,)
Тип данных: float32
✓ Закодировано 3 текстов
